# Customer Retention — Predictive Modeling

The objective of this notebook is to compare multiple classification approaches under the same validation framework.

Performance numbers are deliberately calculated at runtime. Nothing in this notebook assumes a predetermined score.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import TARGET_COLUMN, ID_COLUMN, RANDOM_STATE, CV_SPLITS
from src.modeling import build_candidates

In [ ]:
train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")

def normalize_target(series):
    values = series.astype(str).str.strip().str.lower()
    return values.map({"yes": 1, "no": 0, "1": 1, "0": 0}).astype(int)

y = normalize_target(train[TARGET_COLUMN])
X = train.drop(columns=[TARGET_COLUMN])
X_test = test.copy()

if ID_COLUMN in X.columns:
    X = X.drop(columns=[ID_COLUMN])
if ID_COLUMN in X_test.columns:
    X_test = X_test.drop(columns=[ID_COLUMN])

print(X.shape, y.shape, X_test.shape)

In [ ]:
models = build_candidates(X)

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

results = []

for name, estimator in models.items():
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
        model = clone(estimator)
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model.fit(X_train, y_train)
        probability = model.predict_proba(X_valid)[:, 1]
        score = roc_auc_score(y_valid, probability)
        fold_scores.append(score)

        results.append({
            "model": name,
            "fold": fold,
            "roc_auc": score
        })

    results.append({
        "model": name,
        "fold": "mean",
        "roc_auc": float(np.mean(fold_scores))
    })

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
summary = (
    results_df[results_df["fold"] == "mean"]
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(summary)

plt.figure(figsize=(8, 4))
plt.bar(summary["model"], summary["roc_auc"])
plt.ylabel("Mean validation ROC-AUC")
plt.xlabel("Model")
plt.title("Model comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
best_name = summary.iloc[0]["model"]
best_model = clone(models[best_name])
best_model.fit(X, y)

test_probability = best_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test[ID_COLUMN],
    "Churn": test_probability
})

display(submission.head())
print("Selected model:", best_name)
print("Probability range:", submission["Churn"].min(), submission["Churn"].max())

## Interpretation checklist

Before writing the portfolio case study, record:

1. Which model performed best under the selected metric?
2. How consistent were the fold scores?
3. Did the more complex model materially improve ranking quality?
4. Which engineered variables were useful?
5. What are the practical consequences of false positives and false negatives?
6. What evidence supports the retention recommendations?

Only conclusions supported by the executed experiment should appear in the final case study.